# ElRobot SmolVLA Fine-Tuning

Fine-tune SmolVLA for ElRobot pick & place using LBST/t01_pick_and_place as base.

**Prerequisites:** Upload `smolvla-bundle.zip` (124 MB) from your Pi to this Colab.

**Runtime:** Select GPU → T4 (free tier works)

In [ ]:
# Step 1: Upload the bundle
from google.colab import files
uploaded = files.upload()  # Upload smolvla-bundle.zip

In [ ]:
# Step 2: Unzip and install dependencies
!unzip -qo smolvla-bundle.zip
%cd smolvla-bundle
!pip install uv
!uv sync
print("Dependencies installed.")

In [ ]:
# Step 3: Verify GPU and data
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!ls -lh datasets/*.parquet | wc -l
!echo "Parquet files ready:"
!ls datasets/*.parquet

In [ ]:
# Step 4: Run training
# Fine-tune from LBST pick-and-place checkpoint
# ~5000 steps takes ~30-60 min on T4
!uv run python scripts/train_elrobot.py \
    --steps 5000 \
    --batch-size 32 \
    --lr 1e-4 \
    --warmup-steps 500 \
    --decay-steps 10000 \
    --save-every 1000 \
    --log-every 20 \
    --parquets datasets/*.parquet \
    --base-checkpoint LBST/t01_pick_and_place \
    --output checkpoints/elrobot-run

In [ ]:
# Step 5: Download the trained checkpoint
!ls -lh checkpoints/elrobot-run/final/

# Zip the final checkpoint for download
!cd checkpoints/elrobot-run && zip -r /content/elrobot-checkpoint.zip final/

from google.colab import files
files.download('/content/elrobot-checkpoint.zip')

## After Download

Copy the checkpoint to your Pi 5:
```bash
scp elrobot-checkpoint.zip venay@192.168.137.104:~/
ssh venay@192.168.137.104
cd ~/_Work/Hackathon/norma-core-berlin-hackathon/checkpoints
unzip ~/elrobot-checkpoint.zip -d elrobot-trained
```

Then run inference:
```bash
cd software/ai/smolvla_py
uv run python scripts/run_policy.py \
    --checkpoint ../../checkpoints/elrobot-trained/final \
    --task "pick up the block" \
    --bus-serial 5B61037157 \
    --motor-ids 1,2,3,4,5,6,7,8
```